# Split MNIST Data into 100 Clients
This notebook splits 10,000 MNIST samples into 100 clients, each with 100 data points.
Each client has a balanced distribution: 10 samples from each of the 10 classes (0-9).
The data is saved as separate files in the new_res folder for consistent federated learning experiments.

In [29]:
import numpy as np
from tensorflow.keras.datasets import mnist
import os
from sklearn.utils import shuffle

In [30]:
# Load MNIST dataset
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f"Original training data shape: {x_train.shape}")
print(f"Original training labels shape: {y_train.shape}")
print(f"Original test data shape: {x_test.shape}")
print(f"Original test labels shape: {y_test.shape}")

Loading MNIST dataset...
Original training data shape: (60000, 28, 28)
Original training labels shape: (60000,)
Original test data shape: (10000, 28, 28)
Original test labels shape: (10000,)


In [31]:
# Combine train and test data to have more samples to work with
all_x = np.concatenate([x_train, x_test], axis=0)
all_y = np.concatenate([y_train, y_test], axis=0)

print(f"Combined data shape: {all_x.shape}")
print(f"Combined labels shape: {all_y.shape}")

Combined data shape: (70000, 28, 28)
Combined labels shape: (70000,)


In [32]:
# Shuffle the data
all_x, all_y = shuffle(all_x, all_y, random_state=42)
print("Data shuffled with random_state=42")

Data shuffled with random_state=42


In [33]:
# Select 1000 samples per class (10 classes × 1000 = 10,000 total)
# This allows us to give each of 100 clients exactly 10 samples per class
samples_per_class = 1000
num_classes = 10

x_stratified = []
y_stratified = []

for class_label in range(num_classes):
    # Get indices for this class
    class_indices = np.where(all_y == class_label)[0]
    # Select first 1000 samples of this class
    selected_indices = class_indices[:samples_per_class]
    x_stratified.append(all_x[selected_indices])
    y_stratified.append(all_y[selected_indices])
    print(f"Class {class_label}: Selected {len(selected_indices)} samples")

x_selected = np.concatenate(x_stratified, axis=0)
y_selected = np.concatenate(y_stratified, axis=0)

print(f"\nTotal selected samples: {len(x_selected)}")
print(f"Selected data shape: {x_selected.shape}")
print(f"Selected labels shape: {y_selected.shape}")

Class 0: Selected 1000 samples
Class 1: Selected 1000 samples
Class 2: Selected 1000 samples
Class 3: Selected 1000 samples
Class 4: Selected 1000 samples
Class 5: Selected 1000 samples
Class 6: Selected 1000 samples
Class 7: Selected 1000 samples
Class 8: Selected 1000 samples
Class 9: Selected 1000 samples

Total selected samples: 10000
Selected data shape: (10000, 28, 28)
Selected labels shape: (10000,)


In [34]:
# Verify class distribution
unique, counts = np.unique(y_selected, return_counts=True)
total_samples = len(y_selected)
print("\nClass distribution in selected data:")
for digit, count in zip(unique, counts):
    print(f"Digit {digit}: {count} samples ({count/total_samples*100:.2f}%)")


Class distribution in selected data:
Digit 0: 1000 samples (10.00%)
Digit 1: 1000 samples (10.00%)
Digit 2: 1000 samples (10.00%)
Digit 3: 1000 samples (10.00%)
Digit 4: 1000 samples (10.00%)
Digit 5: 1000 samples (10.00%)
Digit 6: 1000 samples (10.00%)
Digit 7: 1000 samples (10.00%)
Digit 8: 1000 samples (10.00%)
Digit 9: 1000 samples (10.00%)


In [35]:
# Select 500 separate samples for test dataset (50 per class)
# These samples are NOT part of the 10,000 training samples
test_samples_per_class = 50
x_test_stratified = []
y_test_stratified = []

print("\nSelecting 500 test samples (50 per class)...")
for class_label in range(num_classes):
    # Get indices for this class
    class_indices = np.where(all_y == class_label)[0]
    # Skip the first 1000 (already used for training), select next 50
    test_indices = class_indices[samples_per_class:samples_per_class + test_samples_per_class]
    x_test_stratified.append(all_x[test_indices])
    y_test_stratified.append(all_y[test_indices])
    print(f"Class {class_label}: Selected {len(test_indices)} test samples")

x_test_separate = np.concatenate(x_test_stratified, axis=0)
y_test_separate = np.concatenate(y_test_stratified, axis=0)

print(f"\nTotal test samples: {len(x_test_separate)}")
print(f"Test data shape: {x_test_separate.shape}")
print(f"Test labels shape: {y_test_separate.shape}")

# Verify test class distribution
unique, counts = np.unique(y_test_separate, return_counts=True)
print("\nTest data class distribution:")
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count} samples")


Selecting 500 test samples (50 per class)...
Class 0: Selected 50 test samples
Class 1: Selected 50 test samples
Class 2: Selected 50 test samples
Class 3: Selected 50 test samples
Class 4: Selected 50 test samples
Class 5: Selected 50 test samples
Class 6: Selected 50 test samples
Class 7: Selected 50 test samples
Class 8: Selected 50 test samples
Class 9: Selected 50 test samples

Total test samples: 500
Test data shape: (500, 28, 28)
Test labels shape: (500,)

Test data class distribution:
  Digit 0: 50 samples
  Digit 1: 50 samples
  Digit 2: 50 samples
  Digit 3: 50 samples
  Digit 4: 50 samples
  Digit 5: 50 samples
  Digit 6: 50 samples
  Digit 7: 50 samples
  Digit 8: 50 samples
  Digit 9: 50 samples


In [36]:
# Save the complete 10,000 samples dataset
output_dir = 'mnist_100_clients'
os.makedirs(output_dir, exist_ok=True)

complete_dataset_file = os.path.join(output_dir, 'complete_10000_samples.npz')
np.savez(complete_dataset_file, x=x_selected, y=y_selected)
print(f"\nSaved complete dataset: {complete_dataset_file}")
print(f"  - Data shape: {x_selected.shape}")
print(f"  - Labels shape: {y_selected.shape}")

# Save the separate test dataset
test_dataset_file = os.path.join(output_dir, 'test_500_samples.npz')
np.savez(test_dataset_file, x=x_test_separate, y=y_test_separate)
print(f"\nSaved test dataset: {test_dataset_file}")
print(f"  - Data shape: {x_test_separate.shape}")
print(f"  - Labels shape: {y_test_separate.shape}")


Saved complete dataset: mnist_100_clients\complete_10000_samples.npz
  - Data shape: (10000, 28, 28)
  - Labels shape: (10000,)

Saved test dataset: mnist_100_clients\test_500_samples.npz
  - Data shape: (500, 28, 28)
  - Labels shape: (500,)


In [37]:
# Configuration for splitting into clients
num_clients = 100
samples_per_client = 100
samples_per_class_per_client = 10  # Each client gets 10 samples from each class

print(f"\nSplitting into {num_clients} clients with {samples_per_client} samples each...")
print(f"Each client will have {samples_per_class_per_client} samples from each of the 10 classes")


Splitting into 100 clients with 100 samples each...
Each client will have 10 samples from each of the 10 classes


In [38]:
# Split data into clients with stratified distribution
# Each client gets 10 samples from each class
for client_id in range(num_clients):
    client_x = []
    client_y = []
    
    # For each class, give this client 10 samples
    for class_label in range(num_classes):
        # Calculate which samples of this class belong to this client
        class_start_idx = class_label * samples_per_class
        client_class_start = class_start_idx + (client_id * samples_per_class_per_client)
        client_class_end = client_class_start + samples_per_class_per_client
        
        client_x.append(x_selected[client_class_start:client_class_end])
        client_y.append(y_selected[client_class_start:client_class_end])
    
    # Combine all classes for this client
    client_x = np.concatenate(client_x, axis=0)
    client_y = np.concatenate(client_y, axis=0)
    
    # Shuffle client's data
    client_x, client_y = shuffle(client_x, client_y, random_state=client_id)
    
    # Save as .npz file
    filename = os.path.join(output_dir, f'client_{client_id+1}.npz')
    np.savez(filename, x=client_x, y=client_y)
    
    if (client_id + 1) % 10 == 0:
        print(f"Saved {client_id + 1}/{num_clients} clients")

print(f"\nAll {num_clients} client data files saved successfully!")

Saved 10/100 clients
Saved 20/100 clients
Saved 30/100 clients
Saved 40/100 clients
Saved 50/100 clients
Saved 60/100 clients
Saved 70/100 clients
Saved 80/100 clients
Saved 90/100 clients
Saved 100/100 clients

All 100 client data files saved successfully!


In [39]:
# Verify saved data by loading a sample client
test_client_id = 1
test_file = os.path.join(output_dir, f'client_{test_client_id}.npz')
test_data = np.load(test_file)

print(f"\nVerification - Loading client {test_client_id}:")
print(f"Data shape: {test_data['x'].shape}")
print(f"Labels shape: {test_data['y'].shape}")

# Check class distribution
unique, counts = np.unique(test_data['y'], return_counts=True)
print(f"\nClass distribution for client {test_client_id}:")
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count} samples")


Verification - Loading client 1:
Data shape: (100, 28, 28)
Labels shape: (100,)

Class distribution for client 1:
  Digit 0: 10 samples
  Digit 1: 10 samples
  Digit 2: 10 samples
  Digit 3: 10 samples
  Digit 4: 10 samples
  Digit 5: 10 samples
  Digit 6: 10 samples
  Digit 7: 10 samples
  Digit 8: 10 samples
  Digit 9: 10 samples


In [40]:
# Summary statistics
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total training samples: {len(x_selected)}")
print(f"Total test samples: {len(x_test_separate)}")
print(f"Complete training dataset: {output_dir}/complete_10000_samples.npz")
print(f"Separate test dataset: {output_dir}/test_500_samples.npz")
print(f"\nFederated Learning Setup:")
print(f"Number of clients: {num_clients}")
print(f"Samples per client: {samples_per_client}")
print(f"Samples per class per client: {samples_per_class_per_client}")
print(f"Data distribution: Balanced (each client has equal samples from all 10 classes)")
print(f"Client data saved in: {output_dir}/")
print(f"Client files: client_1.npz to client_{num_clients}.npz")
print("="*60)


SUMMARY
Total training samples: 10000
Total test samples: 500
Complete training dataset: mnist_100_clients/complete_10000_samples.npz
Separate test dataset: mnist_100_clients/test_500_samples.npz

Federated Learning Setup:
Number of clients: 100
Samples per client: 100
Samples per class per client: 10
Data distribution: Balanced (each client has equal samples from all 10 classes)
Client data saved in: mnist_100_clients/
Client files: client_1.npz to client_100.npz
